In [1]:
import pandas as pd


import numpy as np

In [2]:
# 1. Leer el CSV
#votaciones = pd.read_csv('hcdn_votaciones_historico.csv')
df = pd.read_csv('votaciones_filtrado.csv')

# 2. Eliminar las columnas inútiles
#votaciones = votaciones.drop(columns=['Unnamed: 0', '¿QUÉ DIJO?'])

# 3. Renombrar las columnas para no tener mayúsculas, espacios ni signos de interrogación
#votaciones = votaciones.rename(columns={
#    'DIPUTADO': 'diputado',
#    'BLOQUE': 'bloque',
#    'PROVINCIA': 'provincia',
#    '¿CÓMO VOTÓ?': 'voto'
#})

# 4. Convertir la fecha (que ahora es texto) a un formato de Fecha real (Datetime)
# Esto es vital para después poder filtrar por "Año" o "Mes" en el dashboard
#votaciones['fecha_votacion'] = pd.to_datetime(votaciones['fecha_votacion'], format='%d/%m/%Y', errors='coerce')

# Vemos el resultado limpio
print(df.head())
print("\nEstructura final del dataset:")
print(df.info())

df.head(8)

                 diputado                        bloque     provincia  \
0  BASTERRA, LUIS EUGENIO  Frente para la Victoria - PJ       Formosa   
1        BIELLA, BERNARDO                U.De.So. Salta         Salta   
2         BREGMAN, MYRIAM     PTS - Frente de Izquierda  Buenos Aires   
3       DEL CAÑO, NICOLAS     PTS - Frente de Izquierda       Mendoza   
4    GARCIA, MARIA TERESA  Frente para la Victoria - PJ  Buenos Aires   

         voto  id_votacion                                    titulo_proyecto  \
0  AFIRMATIVO            1  Régimen previsional especial de carácter excep...   
1  AFIRMATIVO            1  Régimen previsional especial de carácter excep...   
2  AFIRMATIVO            1  Régimen previsional especial de carácter excep...   
3  AFIRMATIVO            1  Régimen previsional especial de carácter excep...   
4  AFIRMATIVO            1  Régimen previsional especial de carácter excep...   

  fecha_votacion           diputado_norm  
0     2015-10-07  BASTERRA, LUI

,diputado,bloque,provincia,voto,id_votacion,titulo_proyecto,fecha_votacion,diputado_norm
0,"BASTERRA, LUIS EUGENIO",Frente para la Victoria - PJ,Formosa,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07,"BASTERRA, LUIS EUGENIO"
1,"BIELLA, BERNARDO",U.De.So. Salta,Salta,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07,"BIELLA, BERNARDO"
2,"BREGMAN, MYRIAM",PTS - Frente de Izquierda,Buenos Aires,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07,"BREGMAN, MYRIAM"
3,"DEL CAÑO, NICOLAS",PTS - Frente de Izquierda,Mendoza,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07,"DEL CANO, NICOLAS"
4,"GARCIA, MARIA TERESA",Frente para la Victoria - PJ,Buenos Aires,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07,"GARCIA, MARIA TERESA"
5,"IANNI, ANA MARIA",Frente para la Victoria - PJ,Santa Cruz,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07,"IANNI, ANA MARIA"
6,"LOUSTEAU, MARTIN",SUMA + UNEN,C.A.B.A.,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07,"LOUSTEAU, MARTIN"
7,"PETRI, LUIS",Unión Cívica Radical,Mendoza,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07,"PETRI, LUIS"


In [ ]:
import re
import pandas as pd

# ============================================================
# PASO 1 — Eliminar ruido procedimental
# ============================================================
#patrones_ruido = [
#    r'APARTAMIENTO DE REGLAMENTO',
#    r'Apartamiento de [Rr]eglamento',
#    r'Apartamiento del [Rr]eglamento',
#    r'A N U L A D A',
#    r'Plan de Labor Parlamentaria',
#    r'Conjunto de proyectos',
#    r'Conjunto de varios',
#    r'^\s*-\s*Votación',
#    r'MOCIÓN SOLICITADA POR',
#    r'Moción solicitada por',
#]
#regex_ruido = '|'.join(patrones_ruido)
#df = votaciones[~votaciones['titulo_proyecto'].str.contains(
#    regex_ruido, regex=True, na=False
#)].copy()
#print(f"[1] Registros originales:          {len(votaciones):,}")
#print(f"[1] Registros tras eliminar ruido: {len(df):,}")
#print(f"[1] Ruido eliminado:               {len(votaciones)-len(df):,}")

# ============================================================
# PASO 2 — Extraer título base + fecha de sesión
# ============================================================
def extraer_titulo_base(titulo):
    if pd.isna(titulo):
        return titulo

    # ── 0. Timestamp al final ─────────────────────────────────────────────────
    titulo = re.sub(r'\s*\d{2}/\d{2}/\d{4}\s*-\s*\d{2}:\d{2}\s*$', '', titulo)

    # ── 1. Formato asterisco (*): "* Artículo N. [texto]" en MEDIO ───────────
    #    Ej: "Exp 60-PE-04 * OD 2696 * Artículo 4. Proyecto de ley..."
    titulo = re.sub(
        r'\s*\*\s*Art[ií]culos?\s*(?:n[°º]\.?\s*|nro\.?\s*)?\d+[°º]?'
        r'(?:\s*(?:bis|ter))?\s*\..*$',
        '', titulo, flags=re.IGNORECASE
    )

    # ── 2. "Titulo N Artículo M. [texto]" en MEDIO ───────────────────────────
    #    Ej: "Expediente 140-S-04 OD 1827 Titulo 4 Artículo 46. Proyecto..."
    titulo = re.sub(
        r'\s+T[ií]tulo\s+\d+\s+Art[ií]culos?\s*\d+\s*\..*$',
        '', titulo, flags=re.IGNORECASE
    )

    # ── 3. Reconsideración de artículo (con dash) ─────────────────────────────
    titulo = re.sub(
        r'\s*[-–]\s*Reconsideraci[oó]n\s+de\s+la\s+votaci[oó]n\s+del\s+Art[ií]culo.*$',
        '', titulo, flags=re.IGNORECASE
    )

    # ── 4. Sufijos formato viejo (separador dash) ─────────────────────────────
    titulo = re.sub(
        r'\s*[-–—]{1,2}\s*'
        r'(Art[ií]culos?\s*(?:n[°º]\.?\s*|nro\.?\s*)?\d+[°º]?'
        r'(?:\s*(?:bis|ter))?(?:\s*(?:a|al|y)\s*\d+)?.*'
        r'|T[ií]tulo\s*\d+.*'
        r'|En\s+(?:General|Particular).*'
        r'|Votaci[oó]n\s+en\s+(?:General|Particular).*)',
        '', titulo, flags=re.IGNORECASE
    )

    # ── 5. Sufijos con punto u otro separador — iterativo hasta convergencia ──
    #    La iteración maneja el caso: ". Dictamen de Mayoría. Artículo 1"
    #    → 1ª pasada: strip ". Artículo 1"
    #    → 2ª pasada: strip ". Dictamen de Mayoría"
    #    También maneja: ". TÍTULO I, CAP. I ART. 1"
    #    → 1ª pasada: strip todo desde TÍTULO (rama nueva)
    #    Y: ". TITULO II. CAPITULO I. ARTS. 2 AL 6"
    #    → 1ª pasada: strip ". ARTS. 2 AL 6"
    #    → 2ª pasada: strip ". CAPITULO I"
    #    → 3ª pasada: strip ". TITULO II"
    _patron = re.compile(
        r'\s*(?:[.·,;*]|[-–—]{1,2})\s*'
        r'('
        r'VOT\.?\s*(?:EN\s+)?(?:GRAL|PART)\.?.*'
        r'|Votaci[oó]n\s+en\s+(?:General|Particular).*'
        r'|DICT\.?\s*DE\s*(?:MAY|MIN)\.?.*'
        r'|Dictamen\s+de\s+(?:Mayor[ií]a|Minor[ií]a)(?:\s+con\s+modificaciones)?\s*(?:I{1,3}|[123])?.*'
        r'|ARTS?\.?\s*\d+(?:\s*(?:AL|A|Y)\s*\d+)?.*'
        r'|Art\.?\s*\d+.*'
        r'|Art[ií]culos?\s*(?:n[°º]\.?\s*|nro\.?\s*)?\d+[°º]?(?:\s*(?:bis|ter))?(?:\s*(?:a|al|y)\s*\d+)?.*'
        r'|CAP[IÍ]TULO\s*\w+.*'
        r'|T[IÍ]TULO\s+(?:[IVXLCDM]+|\d+).*'
        r')$',
        re.IGNORECASE
    )
    prev = None
    while prev != titulo:
        prev = titulo
        titulo = _patron.sub('', titulo)

    # ── 6. Separadores residuales al final ────────────────────────────────────
    titulo = re.sub(r'[\s\*·–—.,;:]+$', '', titulo)
    return titulo.strip()

df['titulo_base'] = df['titulo_proyecto'].apply(extraer_titulo_base)
df['fecha_votacion'] = pd.to_datetime(df['fecha_votacion'])
df['fecha_base'] = df['fecha_votacion'].dt.date

# fecha_sesion = primer día del grupo (diputado, titulo_base)
# resuelve cruces de medianoche
df['fecha_sesion'] = df.groupby(
    ['diputado', 'titulo_base']
)['fecha_base'].transform('min')

print(f"\n[2] Títulos únicos originales: {df['titulo_proyecto'].nunique():,}")
print(f"[2] Títulos base únicos:        {df['titulo_base'].nunique():,}")

# ============================================================
# PASO 3 — Flag de votación "En General"
# ============================================================
patron_general = (
    r'En General|EN GENERAL'           # forma completa
    r'|VOT\.?\s*EN\s*GRAL\.?\s*(Y\s*PART\.?)?'  # VOT. EN GRAL. / VOT. EN GRAL. Y PART.
    r'|VOT\.?\s*GRAL\.?\s*(Y\s*PART\.?)?'       # VOT. GRAL. / VOT GRAL
    r'|EN\s*GRAL\.?'                             # EN GRAL.
)

df['es_voto_general'] = df['titulo_proyecto'].str.contains(
    patron_general, regex=True, na=False
)
print(f"\n[3] Registros con voto 'En General': {df['es_voto_general'].sum():,}")

# ============================================================
# PASO 4 — Consolidación vectorizada
# KEY incluye fecha_sesion para aislar sesiones distintas
# con mismo titulo_base (ej: "Votación en General y Particular...")
# ============================================================
def moda_voto(s):
    m = s.mode()
    if len(m) == 0:    # serie vacía (no debería ocurrir)
        return 'ABSTENCIÓN'
    if len(m) > 1:     # empate real → no podemos determinar posición
        return 'ABSTENCIÓN'
    return m.iloc[0]

cols_contexto = ['bloque', 'provincia', 'fecha_votacion']
KEY = ['diputado', 'titulo_base', 'fecha_sesion']

# 4a — Grupos con voto "En General"
consolidado_general = (
    df[df['es_voto_general']]
    .groupby(KEY)
    .agg(
        voto=('voto', moda_voto),
        **{col: (col, 'first') for col in cols_contexto}
    )
    .reset_index()
    .assign(fuente_consolidacion='en_general')
)

# 4b — Grupos SIN voto "En General"
pares_con_general = set(
    zip(consolidado_general['diputado'],
        consolidado_general['titulo_base'],
        consolidado_general['fecha_sesion'])
)
df_art = df[~df['es_voto_general']].copy()
df_art['_key'] = list(zip(
    df_art['diputado'],
    df_art['titulo_base'],
    df_art['fecha_sesion']
))
df_solo_art = df_art[~df_art['_key'].isin(pares_con_general)].drop(columns='_key')

consolidado_articulos = (
    df_solo_art
    .groupby(KEY)
    .agg(
        voto=('voto', moda_voto),
        **{col: (col, 'first') for col in cols_contexto}
    )
    .reset_index()
    .assign(fuente_consolidacion='moda_articulos')
)

# 4c — Unir y limpiar
df_consolidado = (
    pd.concat([consolidado_general, consolidado_articulos], ignore_index=True)
    .drop(columns=['fecha_sesion'])
)

# ============================================================
# PASO 5 — Eliminar categorías de voto no informativas
# ============================================================
votos_ruido = ['PRESIDENTE', 'PENDIENTE DE INCORPORACIÓN', 'SIN VOTAR']
df_consolidado = df_consolidado[~df_consolidado['voto'].isin(votos_ruido)].copy()

# ============================================================
# VERIFICACIÓN FINAL
# ============================================================
print(f"\n[4] Registros originales:          {len(df):,}")
print(f"[4] Tras consolidar:                {len(df_consolidado):,}")
print(f"[4] Reducción total:                {len(df)-len(df_consolidado):,}")
print(f"\n[4] Fuente de consolidación:")
print(df_consolidado['fuente_consolidacion'].value_counts())
print(f"\n[4] Distribución del voto:")
print(df_consolidado['voto'].value_counts())
print(f"\n[4] Proyectos únicos: {df_consolidado['titulo_base'].nunique():,}")
print(f"[4] Diputados únicos: {df_consolidado['diputado'].nunique():,}")

In [10]:
variantes_general = df['titulo_proyecto'].str.extractall(
    r'(en\s+gral\.?|en\s+grl\.?|vot\.?\s+gral\.?|vot\.?\s+en\s+gral\.?)',
    flags=re.IGNORECASE
)
print(f"Registros con variantes abreviadas de 'En General': {len(variantes_general):,}")
print("\nVariantes encontradas:")
print(variantes_general[0].value_counts().head(20).to_string())

Registros con variantes abreviadas de 'En General': 13,508

Variantes encontradas:
0
VOT. EN GRAL.    11044
VOT EN GRAL.      1150
VOT. GRAL.         343
VOT. GRAL          343
EN GRAL.           192
VOT EN GRAL        191
VOT. EN GRAL        98
VOT GRAL.           49
VOT GRAL            49
Vot. en Gral.       40
en Gral.             5
Vot Gral             2
Vot. Gral.           2


In [11]:
import unicodedata
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
# PARÁMETROS
# ============================================================
THRESHOLD_FUZZY = 0.70
BATCH_SIZE      = 200
URL_PROYECTOS   = ('proyectos_parlamentarios2.1.csv')

# ============================================================
# PASO A — Cargar proyectos desde GitHub
# ============================================================
df_proyectos = pd.read_csv(URL_PROYECTOS)
cols_lookup  = ['TITULO', 'EXP_DIPUTADOS', 'EXP_SENADO', 'AUTOR', 'CAMARA_ORIGEN']
df_proy      = df_proyectos[cols_lookup].copy()
print(f"[A] Proyectos cargados: {len(df_proy):,}")

# ============================================================
# PASO B — Tabla de trabajo sobre titulo_base únicos
# ============================================================
titulos_unicos = (
    df_consolidado[['titulo_base']]
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)
titulos_unicos['autor_final']   = None
titulos_unicos['camara_origen'] = None
titulos_unicos['fuente_autor']  = None
titulos_unicos['score_fuzzy']   = np.nan
print(f"[B] Títulos únicos en df_consolidado: {len(titulos_unicos):,}")

# ============================================================
# PASO C — Funciones de normalización de expedientes
# ============================================================
def normalizar_exp(exp):
    if pd.isna(exp):
        return None
    m = re.match(r'^(\d+)-([A-Z]+)-(\d{2,4})$', str(exp).strip().upper())
    if not m:
        return str(exp).strip().upper()
    num, tipo, año = m.groups()
    if len(año) == 2:
        año = f"19{año}" if int(año) >= 90 else f"20{año}"
    return f"{num}-{tipo}-{año}"

def extraer_exp_de_titulo(titulo):
    for pat in [
        r'(\d{1,5}-D-\d{2,4})',
        r'(\d{1,5}-S-\d{2,4})',
        r'(\d{1,5}-PE-\d{2,4})',
        r'(\d{1,5}-JGM-\d{2,4})',
        r'(\d{1,5}-CD-\d{2,4})',
    ]:
        m = re.search(pat, str(titulo), re.IGNORECASE)
        if m:
            return normalizar_exp(m.group(1))
    return None

# ============================================================
# PASO D — Match determinístico por expediente
# ============================================================
lookup_exp = {}
for _, row in df_proy.iterrows():
    for col in ['EXP_DIPUTADOS', 'EXP_SENADO']:
        exp = normalizar_exp(row[col])
        if exp and exp not in lookup_exp:
            lookup_exp[exp] = (row['AUTOR'], row['CAMARA_ORIGEN'])

titulos_unicos['exp_extraido'] = titulos_unicos['titulo_base'].apply(extraer_exp_de_titulo)

for idx in titulos_unicos[titulos_unicos['exp_extraido'].notna()].index:
    exp = titulos_unicos.at[idx, 'exp_extraido']
    if exp in lookup_exp:
        titulos_unicos.at[idx, 'autor_final']   = lookup_exp[exp][0]
        titulos_unicos.at[idx, 'camara_origen'] = lookup_exp[exp][1]
        titulos_unicos.at[idx, 'fuente_autor']  = 'determinístico'

n_det = (titulos_unicos['fuente_autor'] == 'determinístico').sum()
print(f"[D] Match determinístico: {n_det}/{len(titulos_unicos)} títulos únicos")

# ============================================================
# PASO E — Normalización de texto para TF-IDF
# ============================================================
def normalize_texto(text):
    if pd.isna(text):
        return ""
    text = str(text).upper()
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('ASCII')
    text = re.sub(r'\d{2}/\d{2}/\d{4}\s*-\s*\d{2}:\d{2}', '', text)
    text = re.sub(r'\d{1,5}-[A-Z]+-\d{2,4}', '', text)
    text = re.sub(r'[^A-Z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

# ============================================================
# PASO F — Match fuzzy TF-IDF para los sin autor
# ============================================================
df_fuzzy_input = titulos_unicos[titulos_unicos['fuente_autor'].isna()].copy()
print(f"[F] Títulos para fuzzy: {len(df_fuzzy_input)}")

proy_norm    = df_proy['TITULO'].apply(normalize_texto).tolist()
query_norm   = df_fuzzy_input['titulo_base'].apply(normalize_texto).tolist()

vectorizer   = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
tfidf        = vectorizer.fit_transform(proy_norm + query_norm)
proy_matrix  = tfidf[:len(proy_norm)]
query_matrix = tfidf[len(proy_norm):]

resultados_fuzzy = []
for i in range(0, len(df_fuzzy_input), BATCH_SIZE):
    batch        = query_matrix[i:i + BATCH_SIZE]
    scores       = cosine_similarity(batch, proy_matrix)
    best_indices = np.argmax(scores, axis=1)
    best_scores  = scores[np.arange(len(best_indices)), best_indices]
    for j, (best_idx, best_score) in enumerate(zip(best_indices, best_scores)):
        resultados_fuzzy.append({
            'idx_titulos':  df_fuzzy_input.index[i + j],
            'autor_final':  df_proy.iloc[best_idx]['AUTOR']         if best_score >= THRESHOLD_FUZZY else None,
            'camara_origen':df_proy.iloc[best_idx]['CAMARA_ORIGEN'] if best_score >= THRESHOLD_FUZZY else None,
            'fuente_autor': 'fuzzy'                                  if best_score >= THRESHOLD_FUZZY else None,
            'score_fuzzy':  round(float(best_score), 3),
        })

for r in resultados_fuzzy:
    idx = r['idx_titulos']
    titulos_unicos.at[idx, 'autor_final']   = r['autor_final']
    titulos_unicos.at[idx, 'camara_origen'] = r['camara_origen']
    titulos_unicos.at[idx, 'fuente_autor']  = r['fuente_autor']
    titulos_unicos.at[idx, 'score_fuzzy']   = r['score_fuzzy']

# ============================================================
# PASO G — Join de vuelta a df_consolidado
# ============================================================
df_consolidado = df_consolidado.merge(
    titulos_unicos[['titulo_base', 'autor_final', 'camara_origen',
                    'fuente_autor', 'score_fuzzy']],
    on='titulo_base',
    how='left'
)

# ============================================================
# REPORTE FINAL
# ============================================================
print(f"\n=== COBERTURA AUTOR en df_consolidado ===")
for fuente in ['determinístico', 'fuzzy', None]:
    mask = df_consolidado['fuente_autor'] == fuente if fuente else df_consolidado['fuente_autor'].isna()
    n_tit = df_consolidado[mask]['titulo_base'].nunique()
    n_reg = mask.sum()
    label = fuente if fuente else 'Sin autor'
    print(f"  {label:<20}: {n_tit:>5} títulos únicos | {n_reg:>8,} registros")

print(f"\n  TOTAL con autor: {df_consolidado['autor_final'].notna().sum():,} registros")
print(f"  TOTAL sin autor: {df_consolidado['autor_final'].isna().sum():,} registros")
print(f"\nColumnas en df_consolidado: {df_consolidado.columns.tolist()}")

[A] Proyectos cargados: 112,186
[B] Títulos únicos en df_consolidado: 1,486
[D] Match determinístico: 181/1486 títulos únicos
[F] Títulos para fuzzy: 1305

=== COBERTURA AUTOR en df_consolidado ===
  determinístico      :   181 títulos únicos |    4,806 registros
  fuzzy               :   239 títulos únicos |    7,152 registros
  Sin autor           :  1066 títulos únicos |   32,311 registros

  TOTAL con autor: 11,958 registros
  TOTAL sin autor: 32,311 registros

Columnas en df_consolidado: ['diputado', 'titulo_base', 'voto', 'bloque', 'provincia', 'fecha_votacion', 'fuente_consolidacion', 'autor_final', 'camara_origen', 'fuente_autor', 'score_fuzzy']


In [12]:
# ============================================================
# Tabla de títulos únicos con metadata de autor
# ============================================================
df_titulos_autor = (
    df_consolidado
    .sort_values('fecha_votacion', ascending=False)   # más reciente primero
    .drop_duplicates(subset='titulo_base')
    [['titulo_base', 'autor_final', 'camara_origen',
      'fuente_autor', 'score_fuzzy', 'fecha_votacion']]
    .reset_index(drop=True)
)

df_titulos_autor.to_excel('titulos_autor.xlsx', index=False)

print(f"Filas exportadas: {len(df_titulos_autor):,}")
print(df_titulos_autor.head(5).to_string())

Filas exportadas: 1,486
                                                                                                                   titulo_base autor_final camara_origen fuente_autor  score_fuzzy fecha_votacion
0                                                                                 O.D. 84 - RÉGIMEN DE ZONA FRÍA. MODIFICACIÓN        None          None         None        0.386     2026-05-20
1                                      O.D. 21 - TRATADO DE EXTRADICIÓN ENTRE LA REP. ARGENTINA Y LA REP. DE CHILE. APROBACIÓN        None          None         None        0.375     2026-05-20
2  O.D. 19 - TRATADO SOBRE ASISTENCIA JURÍDICA MUTUA EN MATERIA PENAL, ENTRE LA REP. ARGENTINA Y LA REP. DE SERBIA. APROBACIÓN        None          None         None        0.531     2026-05-20
3                                 O.D. 17 - TRATADO DE EXTRADICIÓN ENTRE LA REP. ARGENTINA Y LA REP. DE COSTA RICA. APROBACIÓN        None          None         None        0.346     2026-05-20
4     

In [9]:
# ============================================================
# EXPORTAR títulos sin autor para completado manual
# ============================================================
sin_autor = (
    df_consolidado[df_consolidado['autor_final'].isna()]
    .sort_values('fecha_votacion', ascending=False)
    .drop_duplicates(subset='titulo_base')
    [['titulo_base', 'fecha_votacion']]
    .reset_index(drop=True)
)
sin_autor['autor_manual']    = ''   # completar a mano
sin_autor['camara_manual']   = ''   # opcional: D / S / PE

sin_autor.to_excel('autores_manuales.xlsx', index=False)
print(f"Títulos sin autor exportados: {len(sin_autor):,}")

Títulos sin autor exportados: 1,215


In [8]:
pd.Series(sorted(df_consolidado['titulo_base'].unique())).to_excel('titulos_consolidado.xlsx', index=False, header=['titulo_base'])